In [ ]:
import numpy as np, pandas as pd
from scipy.ndimage import median_filter

STEP = 0.25            # metres per sample
TARGET = "Frostbumps"

def to_distance_domain(df):
    df = df.sort_values("Loc(m)")
    bins = np.floor((df["Loc(m)"] - df["Loc(m)"].min()) / STEP).astype(int)

    cols = ["CElev(m)", "Roll(deg)", "Pitch(deg)", "AccX", "AccY", "AccZ",
            "GyrX", "GyrY", "GyrZ", "Speed_kmh"]
    out = df[cols].groupby(bins).mean()
    out[TARGET] = df[TARGET].groupby(bins).max()      # any bump sample marks the bin

    out = out.reindex(range(bins.max() + 1))           # fill any empty bins
    out[cols] = out[cols].interpolate(limit_direction="both")
    out[TARGET] = out[TARGET].fillna(0).astype(np.float32)
    out["Loc(m)"] = out.index * STEP + df["Loc(m)"].min()
    return out.reset_index(drop=True)

def add_features(d, k=int(5 / STEP)):          # ~5 m smoothing scale
    f = pd.DataFrame(index=d.index)
    for c in ["AccX", "AccY", "AccZ", "GyrX", "GyrY", "GyrZ"]:
        f[c + "_hp"] = d[c] - median_filter(d[c].values, size=k, mode="nearest")
    f["elev_hp"] = d["CElev(m)"] - median_filter(d["CElev(m)"].values, size=4 * k, mode="nearest")
    f["roll"], f["pitch"], f["speed"] = d["Roll(deg)"], d["Pitch(deg)"], d["Speed_kmh"]
    f["accz_rms"] = f["AccZ_hp"].pow(2).rolling(k, center=True, min_periods=1).mean().pow(0.5)
    return f

def prep(path):
    d = to_distance_domain(pd.read_csv(path))
    return add_features(d), d[TARGET].values

In [ ]:
from sklearn.preprocessing import StandardScaler

train_files = ["../dataset/cleaned/Iisingintie0.csv", "../dataset/cleaned/Iisingintie1.csv", "../dataset/cleaned/Sinetänsalmentie0.csv", "../dataset/cleaned/Sinetänsalmentie1.csv", "../dataset/cleaned/Sinetänsalmentie2.csv"]
val_files   = ["../dataset/cleaned/Sinetänsalmentie3.csv"]
test_files  = ["../dataset/cleaned/Iisingintie2.csv"]

def load(file_list):
    return [prep(f) for f in file_list]    # list of (features, labels), one per file

train, val, test = load(train_files), load(val_files), load(test_files)

# Fit scaler on TRAIN ONLY (concatenate the feature tables just for fitting)
scaler = StandardScaler().fit(pd.concat([f for f, _ in train]))

def make_windows(f, y, win=256, stride=64):
    X = scaler.transform(f)
    Xs, ys = [], []
    for i in range(0, len(X) - win + 1, stride):
        Xs.append(X[i:i+win])
        ys.append(y[i:i+win])
    return np.array(Xs, np.float32), np.array(ys, np.float32)[..., None]

def build(split, stride):
    parts = [make_windows(f, y, stride=stride) for f, y in split]
    return np.concatenate([p[0] for p in parts]), np.concatenate([p[1] for p in parts])

Xtr, ytr = build(train, stride=64)     # overlapping windows = more training samples
Xva, yva = build(val,   stride=256)    # non-overlapping, so metrics aren't double counted
Xte, yte = build(test,  stride=256)

print(Xtr.shape, Xva.shape, Xte.shape)
print("bump fraction:", ytr.mean(), yva.mean(), yte.mean())

(457, 256, 11) (7, 256, 11) (49, 256, 11)
bump fraction: 0.22907549 0.390625 0.18335459
